# Cointegration Testing

Imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import statsmodels.api as sm
from statsmodels.tsa.stattools import coint

### Step 1: Define formation window and extract data
We will define 2016-01-01 as the end of the formation window and extract the previous 2 years of trading data (504 trading days).

In [2]:
formation_end = pd.Timestamp("2016-01-01")
window_size = 504

In [3]:
data_folder_path = Path("data")

membership_file_path = data_folder_path / "membership_df.csv"

print(data_folder_path.exists())

True


Extract list of S&P tickers at formation end

In [4]:
formatted_date = formation_end.strftime('%Y%m')
end_membership = pd.read_csv(membership_file_path)[formatted_date].dropna().to_list()

In [5]:
print(end_membership)

['A', 'AA', 'AAL', 'AAP', 'AAPL', 'ABBV', 'ABT', 'ACN', 'ADBE', 'ADI', 'ADM', 'ADP', 'ADS', 'ADSK', 'ADT', 'AEE', 'AEP', 'AES', 'AET', 'AFL', 'AGN', 'AIG', 'AIV', 'AIZ', 'AKAM', 'ALL', 'ALLE', 'ALXN', 'AMAT', 'AME', 'AMG', 'AMGN', 'AMP', 'AMT', 'AMZN', 'AN', 'ANDV', 'AON', 'APA', 'APC', 'APD', 'APH', 'APTV', 'ARG', 'ATVI', 'AVB', 'AVGO', 'AVY', 'AXP', 'AZO', 'BA', 'BAC', 'BALL', 'BAX', 'BBBY', 'BBWI', 'BBY', 'BCR', 'BDX', 'BEN', 'BF-B', 'BHI', 'BIIB', 'BK', 'BKNG', 'BLK', 'BMY', 'BRCM', 'BRK-B', 'BSX', 'BWA', 'BXLT', 'BXP', 'C', 'CA', 'CAG', 'CAH', 'CAM', 'CAT', 'CB', 'CBRE', 'CCE', 'CCI', 'CCL', 'CELG', 'CERN', 'CF', 'CHD', 'CHK', 'CHRW', 'CI', 'CINF', 'CL', 'CLX', 'CMA', 'CMCSA', 'CME', 'CMG', 'CMI', 'CMS', 'CNP', 'CNX', 'COF', 'COL', 'COP', 'COR', 'COST', 'CPAY', 'CPB', 'CPGX', 'CPRI', 'CRM', 'CSCO', 'CSRA', 'CSX', 'CTAS', 'CTRA', 'CTSH', 'CTXS', 'CVC', 'CVS', 'CVX', 'D', 'DAL', 'DAY', 'DD', 'DE', 'DFS', 'DG', 'DGX', 'DHI', 'DHR', 'DIS', 'DISCA', 'DISCK', 'DLTR', 'DNB', 'DO', 'DOC',

Get price data for window

In [6]:
historical_ohlcv_df = pd.read_parquet(data_folder_path / "historical_ohlcv_df.parquet")

In [7]:
print(historical_ohlcv_df.head())

Ticker     SOLV                       KVUE                        ...  \
Price      Open High Low Close Volume Open High Low Close Volume  ...   
Date                                                              ...   
2014-01-02  NaN  NaN NaN   NaN    NaN  NaN  NaN NaN   NaN    NaN  ...   
2014-01-03  NaN  NaN NaN   NaN    NaN  NaN  NaN NaN   NaN    NaN  ...   
2014-01-06  NaN  NaN NaN   NaN    NaN  NaN  NaN NaN   NaN    NaN  ...   
2014-01-07  NaN  NaN NaN   NaN    NaN  NaN  NaN NaN   NaN    NaN  ...   
2014-01-08  NaN  NaN NaN   NaN    NaN  NaN  NaN NaN   NaN    NaN  ...   

Ticker              R                                                 EOG  \
Price            Open       High        Low      Close  Volume       Open   
Date                                                                        
2014-01-02  51.244708  51.384891  50.683965  50.915272  453500  59.113503   
2014-01-03  51.055471  51.335842  50.838185  51.020428  303500  58.822462   
2014-01-06  51.265752  51.4479

Extract data in formation window

In [8]:
formation_data = formation_data = historical_ohlcv_df.loc[
    historical_ohlcv_df.index < formation_end
].tail(window_size)

Keep only tickers present at formation end

In [9]:
formation_data = formation_data.loc[:, formation_data.columns.get_level_values(0).isin(end_membership)]

Extract close prices for cointegration

In [10]:
formation_close_df = formation_data.xs('Close', axis=1, level=1)

In [11]:
print(formation_close_df.head())

Ticker      SNI  LLL  WBA       CBRE  ADT        IQV      KLAC        SW  \
Date                                                                       
2014-01-02  NaN  NaN  NaN  26.340000  NaN  45.799999  3.997444  16.12015   
2014-01-03  NaN  NaN  NaN  26.400000  NaN  46.290001  4.003122  16.12015   
2014-01-06  NaN  NaN  NaN  26.250000  NaN  46.029999  3.933112  16.12015   
2014-01-07  NaN  NaN  NaN  26.190001  NaN  46.790001  3.931221  16.12015   
2014-01-08  NaN  NaN  NaN  26.190001  NaN  47.419998  3.929329  16.12015   

Ticker            AMT  SYF  ...  AGN        CME        GIS        CAG  \
Date                        ...                                         
2014-01-02  58.898232  NaN  ...  NaN  47.224113  31.121460  16.424904   
2014-01-03  59.068741  NaN  ...  NaN  47.975243  31.033251  16.385775   
2014-01-06  59.446808  NaN  ...  NaN  47.490654  31.083658  16.307476   
2014-01-07  60.269669  NaN  ...  NaN  47.139320  31.404943  16.586372   
2014-01-08  60.744110  NaN  .

Get number of columns before removing any

In [12]:
num_cols_before = len(formation_close_df.columns)
print(num_cols_before)

508


Remove columns where <= 90% of the expected observation data is present

In [13]:
min_observations = int(len(formation_close_df) * 0.90)

formation_close_df = formation_close_df.dropna(
    axis=1,
    thresh=min_observations
)

In [14]:
num_cols_after = len(formation_close_df.columns.tolist())
print(f"{num_cols_after/num_cols_before:.2%} of columns remain after dropping columns with less than {min_observations} observations.")

77.56% of columns remain after dropping columns with less than 453 observations.
